# Richardson et al. (2018) on OpenNeuro ds000228, snapshot 1.1.1

Paper: Richardson, Lisandrelli, Riobueno-Naylor & Saxe (2018), "Development of the social brain
from age three to twelve years", Nature Communications 9, 1027,
[doi:10.1038/s41467-018-03399-2](https://doi.org/10.1038/s41467-018-03399-2).

Dataset: [OpenNeuro ds000228](https://openneuro.org/datasets/ds000228/versions/1.1.1), snapshot
`1.1.1`, DOI 10.18112/openneuro.ds000228.v1.1.1. The paper cites the dataset on OpenfMRI
without a version; the claims file names snapshot 1.1.1, which is also the latest published
snapshot. Its CHANGES file says 1.1.1 only updated the citation to the paper (2023), so the
pinned snapshot holds the same data as 1.1.0. It is pinned because the claims file gives it and
the protocol pins the latest published version when the paper cites none.

Claims: `claims/openneuro-ds000228/10.1038-s41467-018-03399-2.yaml`, written from the version of
record and its Supplementary Information (Supplementary Tables 1 to 4 read from the PDF). Its
header labels the shared pipeline P1 to P9 and the open details O1 to O9; this notebook uses the
same labels.

What the snapshot holds (listed below from the archive): for all 155 participants the raw BOLD
and T1 images and the paper's own preprocessed data (SPM8, normalized, smoothed; ART artifact
indicators and five CompCor regressors per run; white-matter and brain masks), the group ROIs
of every network the paper uses (ToM, Pain, face and scene parcels, M1, V1), and the behavioural
scores in participants.tsv. Every file is read into memory from its URL pinned to the snapshot.

| Claim | Tested here |
|---|---|
| participants, supp-table1-behavior-by-age, beh-tom-age, beh-tom-dccs, beh-fb-groups, beh-fb-tom, beh-fb-age, beh-fb-dccs | tested (participants.tsv) |
| motion-child-adult, motion-not-age-tom, motion-not-fb | tested (deposited ART indicators) |
| irc-adult, irc-age-within, irc-age-across, irc-adult-vs-child, irc-3yo-distinct, irc-3yo-not-anticorrelated, irc-3yo-homologous, irc-3yo-nonbilateral, irc-tom-behavior, irc-tom-behavior-age, irc-fb-within, irc-fb-within-dccs, irc-fb-across | tested (deposited preprocessed BOLD and ROIs) |
| irc-control-networks, irc-williams, supp-table4-other-networks, face-rc, face-overlap | tested; the face and scene parcels the claims file lists as outside input are deposited in the snapshot |
| rc-events, rc-overlap, rc-child-adult-timecourse, rc-age-events, rc-t04-tom, rc-fb-no-difference, rc-3yo-events | tested |
| maturity-across, maturity-within, maturity-tom-behavior, maturity-fb | tested |
| replication-sample | blocked: the independent sample of 20 adults is not deposited anywhere |
| hand-coded-events | blocked: the hand coding of Jacoby et al. 2016 is not in the snapshot or the paper, and the lab's script that holds it is not pinned |

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import social_brain as sb

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dataset", sb.DATASET_ID, "snapshot", sb.SNAPSHOT, "doi", sb.SNAPSHOT_DOI)
files = sb.file_table()
print(len(files), "files,", round(files["size"].sum() / 1e9, 2), "GB")
kinds = files.index.str.replace(r"sub-pixar\d+", "sub-*", regex=True)
print(files.groupby(kinds)["size"].agg(["count", "sum"]).rename(columns={"sum": "bytes"}).to_string())

In [ ]:
# The files read whole, with their checksums (OpenNeuro `id`); the per-participant files read are
# the swrf BOLD image and the nuisance-regressor file of each participant, listed after extraction.
for f in ["dataset_description.json", "README", "CHANGES", "participants.tsv", "participants.json",
          "task-pixar_bold.json", "derivatives/README.txt"]:
    print(f, files.loc[f, "id"])
for r in sb.ALL_ROIS:
    f = [k for k in files.index if k.startswith("derivatives/ROIs/" + r)]
    print(f[0], files.loc[f[0], "id"])
print()
print(sb.text("CHANGES"))
print(sb.text("README"))

In [ ]:
print(sb.text("derivatives/README.txt"))
print(sb.text("participants.json"))
print(sb.text("task-pixar_bold.json"))

## Loading

Each participant's preprocessed image (`*_swrf_bold.nii.gz`, about 90 MB) is read into memory,
and the voxels of the 29 deposited ROIs are kept; the nuisance file gives the artifact
indicators and the CompCor regressors. This takes about two minutes.

In [ ]:
p = sb.participants()
subs = list(p.participant_id)
D = sb.extract_all(subs, n_threads=8)
p["n_art"] = [len(sb.artifact_index(D[s]["artifacts"])) for s in subs]
p = p.set_index("participant_id")
TOM, NOFB, DCCS = "ToM Booklet-Matched", "ToM Booklet-Matched-NOFB", "DCCS Summary"
children = p[p.Child_Adult == "child"].copy()
adults = p[p.Child_Adult == "adult"].copy()
young = children[children.Age < 6].copy()
print(len(subs), "participants loaded")
print(pd.Series({s: files.loc[sb.derivative(s, "task-pixar_run-001_swrf_bold.nii.gz"), "id"]
                 for s in subs[:3]}).to_string(), "... (checksums of all 155 in `files`)")

## Data check

In [ ]:
print("participants.tsv columns:", list(p.reset_index().columns))
print(p.reset_index().describe(include="all").T.to_string())
print()
print("children", len(children), "adults", len(adults))
print("children: female", (children.Gender == "F").sum(), "mean age %.2f (s.d. %.2f)" % (children.Age.mean(), children.Age.std()),
      "range %.2f to %.2f" % (children.Age.min(), children.Age.max()))
print("adults: female", (adults.Gender == "F").sum(), "mean age %.2f (s.d. %.2f)" % (adults.Age.mean(), adults.Age.std()),
      "range %.0f to %.0f" % (adults.Age.min(), adults.Age.max()))
print(p.groupby("AgeGroup").Age.agg(["count", "min", "max", "mean", "std"]).round(2).to_string())
print("handedness:", p.groupby("Child_Adult").Handedness.value_counts().to_dict())
print("under 6:", len(young), "FB groups", young.FB_Group.value_counts().to_dict(),
      "; FB_Group in children aged 6 and over:", children[children.Age >= 6].FB_Group.value_counts().to_dict())
print("DCCS present:", children[DCCS].notna().sum(), "children, all aged under 6:", bool((children.loc[children[DCCS].notna(), "Age"] < 6).all()))
print("FB_Composite against FB_Group (cut-offs of the Methods):")
print(pd.crosstab(young.FB_Composite, young.FB_Group).to_string())
print("ToM score range %.2f to %.2f; no-FB score range %.2f to %.2f" % (children[TOM].min(), children[TOM].max(), children[NOFB].min(), children[NOFB].max()))
print()
print("voxel size by age group (Methods: 3.13 mm for 5 + 28 adults, 3 7yo, 20 8-12yo):")
print(pd.crosstab(p.AgeGroup, [p["Scanlog: Voxel slize"], p["Scanlog: Slice Gap"].fillna(0)]).to_string())
print("coil:", pd.crosstab(p.AgeGroup, p["Scanlog: Coil"]).to_dict())

In [ ]:
shapes = {tuple(D[s]["shape"]) for s in subs}
zooms = {tuple(float(z) for z in D[s]["zooms"][:3]) for s in subs}
print("preprocessed image shapes:", shapes, "voxel sizes (mm):", zooms, "dtype:", {D[s]["dtype"] for s in subs})
print("affines identical:", all(np.allclose(D[s]["affine"], D[subs[0]]["affine"]) for s in subs))
print("168 volumes in every image (TR 2 s from task-pixar_bold.json):", all(D[s]["shape"][3] == sb.N_TR for s in subs))
print("artifact indicator rows each flag one timepoint:", all((D[s]["artifacts"].sum(axis=1) == 1).all() for s in subs))
print("CompCor rows per participant:", {D[s]["compcor"].shape for s in subs})
print("artifact timepoints per participant: max %d of 168 (%.2f; Methods drop at one third)" % (p.n_art.max(), p.n_art.max() / 168))
print(p.groupby("AgeGroup").n_art.agg(["mean", "std", "min", "max"]).round(2).to_string())
roi_size = pd.Series({r: D[subs[0]]["rois"][r]["n_vox"] for r in sb.ALL_ROIS})
print("ROI sizes (voxels):", roi_size.to_dict())
print("Supplementary Table 2 sizes: RTPJ 376 LTPJ 368 PC 382 DMPFC 217 MMPFC 275 VMPFC 198; RS2 368 LS2 269 RInsula 309 LInsula 240 RMFG 142 LMFG 256 AMCC 249; RFFA 1019 LFFA 531")
cover = pd.DataFrame({s: {r: D[s]["rois"][r]["n_with_data"] for r in sb.ALL_ROIS} for s in subs}).T
print("participants with any ROI voxel without data (zero or non-finite at some timepoint), per ROI:")
print((cover < roi_size).sum().to_string())
print("fewest voxels with data in a face ROI:", int(cover[sb.FACE_ROIS].min().min()),
      "(Supplementary Fig. 4: every participant had at least 100 voxels in every face ROI)")
face_incomplete = (cover[sb.FACE_ROIS] < roi_size[sb.FACE_ROIS]).any(axis=1)
print("participants with incomplete face-ROI coverage:", int(face_incomplete.sum()),
      p.loc[face_incomplete[face_incomplete].index, "AgeGroup"].value_counts().to_dict(),
      "(Supplementary Fig. 4: 17, of them 2 adults, 12 8-12yo, one each 7yo, 4yo, 3yo)")
print("NIfTI scl_slope, scl_inter:", {(str(D[s]["scl_slope"]), str(D[s]["scl_inter"])) for s in subs})
rng_img = pd.DataFrame({s: D[s]["range"] for s in subs}, index=["min", "max"]).T
print("image value range: minimum %.1f to %.1f, maximum %.1f to %.1f over participants; %s %s" % (
    rng_img["min"].min(), rng_img["min"].max(), rng_img["max"].min(), rng_img["max"].max(), subs[0], D[subs[0]]["range"]))
# ROI coordinates: centroid and largest distance from the stated center, to check that the masks
# are in MNI mm (derivatives/README.txt and Supplementary Table 2 centers; M1 and V1 from Methods).
STATED = {"RTPJ": (48, -60, 30), "LTPJ": (-48, -62, 30), "PC": (0, -54, 34), "DMPFC": (-6, 54, 36), "MMPFC": (-4, 58, 16),
          "VMPFC": (-4, 56, -16), "RS2": (60, -28, 38), "LS2": (-62, -32, 34), "RInsula": (42, 6, -6), "LInsula": (-42, -2, -4),
          "RMFG": (50, 42, 12), "LMFG": (-46, 36, 14), "AMCC": (0, 2, 42), "rFFA": (38, -42, -22), "lFFA": (-40, -52, -18),
          "right_primotor": (38, -24, 58), "left_primotor": (-38, -20, 58), "right_CS": (10, -86, 2), "left_CS": (-10, -86, 2)}
cen = []
for r in sb.ALL_ROIS:
    xyz = sb.all_roi_mm()[r]
    row = {"roi": r, "n": xyz.shape[1], "centroid": tuple(np.round(xyz.mean(axis=1), 1))}
    if r in STATED:
        dist = np.linalg.norm(xyz - np.array(STATED[r])[:, None], axis=0)
        row.update({"stated center": STATED[r], "centroid to center mm": round(float(np.linalg.norm(xyz.mean(axis=1) - STATED[r])), 1),
                    "max distance mm": round(float(dist.max()), 1)})
    cen.append(row)
print(pd.DataFrame(cen).to_string(index=False))


Timing check. There are no events files; the movie timing is fixed (README: TRs 1 to 5 rest,
6 to 10 logos, movie from TR 11). Two checks that the image volumes and the regressors share the
same clock: (1) the ROI signal should deviate most at isolated artifact timepoints, compared
with the same indicators shifted in time; (2) the adult ToM-network signal should be
higher during the paper's ToM events than during its pain events (Supplementary Table 3), and the
contrast should drop when the event labels are shifted in time; this is shown in the reverse
correlation section, after the network timecourses are built.

In [ ]:
def spike_at(s, shift):
    # isolated artifact timepoints only; deviation of the mean ToM+Pain ROI signal from the median
    # of the 7 timepoints around it, in units of that participant's median deviation
    x = np.mean([D[s]["rois"][r]["all"] for r in sb.TOM_ROIS + sb.PAIN_ROIS], axis=0)
    med = pd.Series(x).rolling(7, center=True, min_periods=4).median().values
    dev = np.abs(x - med); dev = dev / np.median(dev)
    a = sb.artifact_index(D[s]["artifacts"])
    iso = [t for t in a if t - 1 not in a and t + 1 not in a and 3 <= t + shift < sb.N_TR - 3]
    return [dev[t + shift] for t in iso]
for shift in [-2, -1, 0, 1, 2, 10]:
    vals = np.concatenate([spike_at(s, shift) for s in subs])
    print(f"isolated artifact timepoints shifted by {shift:3d} TRs: n = {len(vals)}, median deviation = {np.median(vals):.2f}")

## Behaviour (participants.tsv)

### participants

> The sample is 122 children aged 3.5 to 12 years (64 female) and 33 adults aged 18 to 39 (20
> female); five further children were excluded for excessive motion. By age group there are 17,
> 14, 34, 23 and 34 children aged 3, 4, 5, 7 and 8 to 12 years.

In [ ]:
counts = {"children": len(children), "adults": len(adults), "female children": int((children.Gender == "F").sum()),
          "female adults": int((adults.Gender == "F").sum())}
counts.update({f"children {g}": int((children.AgeGroup == g).sum()) for g in sb.AGE_GROUPS})
counts["mean child age"] = round(children.Age.mean(), 2)
expected = [122, 33, 64, 20, 17, 14, 34, 23, 34, 6.7]
print(pd.DataFrame({"measured": pd.Series(counts), "expected": expected}).to_string())
print("Age-group ranges:", children.groupby("AgeGroup").Age.agg(["min", "max"]).round(2).to_dict("index"))
print("The 5 children excluded for motion are not in the dataset (155 participants listed).")

**Result: reproduced.** All nine counts match exactly (122 children, 33 adults, 64 and 20 female; 17, 14, 34, 23 and 34 children by age group) and the mean child age is 6.71 years against 6.7. The age-group ranges match Supplementary Table 1. The five children excluded for motion are not in the dataset.

### supp-table1-behavior-by-age

> Mean ToM score rises across age groups (3yo .54, 4yo .63, 5yo .73, 7yo .88, 8-12yo .96), and
> the false-belief groups (pass/inconsistent/fail) are 4/4/9 at 3 years, 3/7/4 at 4 years and
> 23/9/2 at 5 years.

In [ ]:
m = children.groupby("AgeGroup")[TOM].agg(["mean", "std", "count"]).loc[sb.AGE_GROUPS]
m["expected"] = [0.54, 0.63, 0.73, 0.88, 0.96]
m["within 10%"] = (m["mean"] - m.expected).abs() <= 0.1 * m.expected
print(m.round(3).to_string())
fb = pd.crosstab(young.AgeGroup, young.FB_Group)[["pass", "inc", "fail"]]
print(fb.to_string())
print("expected 3yo 4/4/9, 4yo 3/7/4, 5yo 23/9/2; totals", fb.sum().to_dict(), "(expected 30, 20, 15)")

**Result: reproduced.** The five mean ToM scores (.517, .610, .724, .875, .956) are within 10% of .54, .63, .73, .88 and .96, and the nine false-belief group counts by age (4/4/9, 3/7/4, 23/9/2) match exactly.

### beh-tom-age

> Among children, ToM score (proportion correct) increases with age (Kendall tau = .66), and
> among the 3- to 5-year-olds DCCS summary score also increases with age (Kendall tau = .20).

In [ ]:
t1 = sb.kendall(children[TOM], children.Age)
yd = young.dropna(subset=[DCCS])
t2 = sb.kendall(yd[DCCS], yd.Age)
print(f"ToM score ~ age, n = {len(children)}: tau-b = {t1[0]:.3f}, p = {t1[1]:.2g} (paper .66, p < .00001)")
print(f"DCCS ~ age, n = {len(yd)}: tau-b = {t2[0]:.3f}, p = {t2[1]:.3f} (paper .20, p = .049)")
for v in ["c"]:
    print(f"  variant tau-{v}:", round(stats.kendalltau(children[TOM], children.Age, variant=v).statistic, 3),
          round(stats.kendalltau(yd[DCCS], yd.Age, variant=v).statistic, 3))

**Result: reproduced.** ToM score against age: tau-b = .665 (paper .66), p = 5.4e-26. DCCS against age in the 64 children with a DCCS score: tau-b = .198 (paper .20), p = .048 (paper .049, same side of .05).

### beh-tom-dccs

> In 3- to 5-year-olds, ToM and DCCS scores are positively correlated controlling for age
> (partial Kendall tau = .19).

In [ ]:
r, pv = sb.partial_kendall(yd[TOM], yd[DCCS], yd.Age)
print(f"partial tau(ToM, DCCS | age), n = {len(yd)}: {r:.3f}, p = {pv:.3f} (paper .19, p = .03)")

**Result: reproduced.** Partial Kendall tau of ToM and DCCS given age = .186 (paper .19, tolerance .019), p = .031 (paper .03).

### beh-fb-groups

> Among the 65 children aged 3 to 5, the false-belief composite gives 30 passers (15 female), 20
> inconsistent performers (13 female) and 15 failers (6 female).

In [ ]:
g = young.groupby("FB_Group").agg(n=("Age", "size"), female=("Gender", lambda x: int((x == "F").sum())),
                                  age_mean=("Age", "mean"), age_sd=("Age", "std"))
print(g.round(2).to_string())
recut = pd.cut(young.FB_Composite, [-1, 2, 4, 6], labels=["fail", "inc", "pass"]).astype(str)
print("groups recomputed from FB_Composite agree with FB_Group:", bool((recut == young.FB_Group).all()))
print("handedness by group:", young.groupby("FB_Group").Handedness.value_counts().to_dict())

**Result: reproduced.** 30 passers (15 female), 20 inconsistent (13 female), 15 failers (6 female), as deposited in FB_Group; the groups recomputed from FB_Composite with the Methods' cut-offs are the same.

### beh-fb-tom, beh-fb-age, beh-fb-dccs

> False-belief failers and inconsistent performers score lower than passers on the remaining ToM
> items (means .55, .57 and .75). False-belief failers are younger than inconsistent performers
> and passers (mean ages 4.1, 4.8 and 5.2 years). False-belief failers have lower DCCS scores than
> the other two groups (means 1.73, 2.26, 2.33).

Tukey HSD over the three groups (statsmodels `pairwise_tukeyhsd`), diffs under both scales of O9;
Kruskal-Wallis from scipy.

In [ ]:
def fb_tests(var):
    sub = young.dropna(subset=[var])
    out = sub.groupby("FB_Group")[var].agg(["mean", "sem", "count"]).loc[["fail", "inc", "pass"]]
    print(out.round(3).to_string())
    kw = stats.kruskal(*[sub.loc[sub.FB_Group == g, var] for g in ["pass", "inc", "fail"]])
    print(f"Kruskal-Wallis H = {kw.statistic:.2f}, p = {kw.pvalue:.2g}")
    for scale in ["z-scored (O9a)", "raw (O9b)"]:
        v = (sub[var] - sub[var].mean()) / sub[var].std() if scale.startswith("z") else sub[var]
        t = pairwise_tukeyhsd(v.values, sub.FB_Group.values)
        df = pd.DataFrame(t.summary().data[1:], columns=t.summary().data[0])
        df["p-adj (exact)"] = [f"{x:.2g}" for x in t.pvalues]
        print(" Tukey,", scale); print(df.to_string(index=False))
    return out, kw

print("== beh-fb-tom, reading (a): 18 non-FB items (ToM Booklet-Matched-NOFB)")
fbtom_a = fb_tests(NOFB)
print("\n== beh-fb-tom, reading (b): full ToM score")
fbtom_b = fb_tests(TOM)

**Result: reproduced.** Under reading (a), the 18 non-false-belief items deposited as ToM Booklet-Matched-NOFB, the means are .546, .572 and .753 (paper .55, .57, .75) and Kruskal-Wallis H = 22.95 (paper 22.96), p = 1e-05. The Tukey tests match: Pass-Fail diff 1.24 on the z scale (paper 1.2), p = 3.8e-05; Pass-Inc 1.08 (paper 1.08), p = 7.8e-05; Inc-Fail .16 (paper .16), p = .85. The z-scaled diffs (O9a) are the ones that match the paper's. Reading (b), the full score, gives means .451, .571, .792, out of tolerance for failers and passers.

In [ ]:
fbage = fb_tests("Age")

**Result: reproduced.** Mean ages 4.08, 4.75 and 5.23 (paper 4.1, 4.8, 5.2). Tukey on the z scale: Pass-Fail diff 1.42 (paper 1.4), p = 4.3e-06; Inc-Fail .83 (paper .83), p = .015 (paper .01); Pass-Inc .59 (paper .59), p = .047 (paper .047).

In [ ]:
fbdccs = fb_tests(DCCS)

**Result: reproduced.** Mean DCCS 1.733, 2.263 and 2.333 (paper 1.73, 2.26, 2.33), Kruskal-Wallis H = 7.56, p = .023 (paper 7.56, .02); Tukey Pass-Fail diff .88 on the z scale (paper .88), p = .012; Inc-Fail .78 (paper .78), p = .052 (paper .052, reported, not deciding).

## Motion

The number of artifact timepoints is the number of ART indicator rows in each participant's
deposited nuisance file (P2).

### motion-child-adult

> Children have more artifact timepoints than adults (mean 10.5 against 2.8).

In [ ]:
w = stats.ttest_ind(children.n_art, adults.n_art, equal_var=False)
print(p.groupby("Child_Adult").n_art.agg(["mean", "std", "count"]).round(2).to_string())
print(f"Welch t = {w.statistic:.2f}, df = {w.df:.1f}, p = {w.pvalue:.2g} (paper t(137.7) = 6.49, p < .000001)")

**Result: reproduced.** Mean artifact timepoints 10.51 (s.d. 10.6) in children and 2.82 (s.d. 4.0) in adults (paper 10.5 and 2.8); Welch t = 6.49 on 137.7 df, p = 1.4e-09, as in the paper.

### motion-not-age-tom

> Among children, the number of artifact timepoints is not correlated with age or with ToM score.

In [ ]:
rs = stats.spearmanr(children.n_art, children.Age)
rk = sb.kendall(children.n_art, children[TOM])
print(f"Spearman(n_art, age) = {rs.statistic:.3f}, p = {rs.pvalue:.2f} (paper .02, p = .86)")
print(f"Kendall(n_art, ToM score) = {rk[0]:.3f}, p = {rk[1]:.2f} (paper -.005, p = .94)")

**Result: reproduced.** Neither correlation is significant: Spearman with age .017, p = .86; Kendall with ToM score -.005, p = .94, the paper's values.

### motion-not-fb

> In 3- to 5-year-olds, the number of artifact timepoints does not differ between false-belief
> passers and failers, nor vary with DCCS score.

In [ ]:
pf = young[young.FB_Group.isin(["pass", "fail"])].copy()
pf["fail"] = (pf.FB_Group == "fail").astype(float)
for z in [False, True]:
    r1 = sb.ols(pf, "n_art", ["fail"], zscore=z)
    r2 = sb.ols(yd, "n_art", [DCCS], zscore=z)
    print(f"{'z-scored (O4a)' if z else 'raw (O4b)'}: FB group (fail = 1, n = {int(r1.nobs)}) b = {r1.params['fail']:.3f}, t = {r1.tvalues['fail']:.2f}, p = {r1.pvalues['fail']:.2f};"
          f" DCCS (n = {int(r2.nobs)}) b = {r2.params[DCCS]:.3f}, t = {r2.tvalues[DCCS]:.2f}, p = {r2.pvalues[DCCS]:.2f}")
print("paper: FB b = -.04, t = -.12, p = .9; DCCS b = .16, t = 1.18, p = .25")

**Result: reproduced.** Neither effect is significant: FB group t = .12, p = .90 (z-scaled b = .039; the paper's sign is the opposite coding), DCCS t = 1.17, p = .24 (z-scaled b = .148, paper .16).

## Inter-region correlation

Pipeline P4 and P5 on the deposited preprocessed images. For each participant, every ROI
timecourse is cleaned (`sb.clean`) and correlated with every other. Readings run: O1 (all voxels
of the deposited masks, `all`, or only voxels with data, `with_data`; the deposited masks match
Supplementary Table 2 so reading O1b, full spheres, is not needed), O2 (one indicator per
artifact timepoint, a, or one regressor, b) and O3 (all 168 TRs, a, or TRs 11 to 168, b). The
main reading is O1 all, O2a (the deposited regressors), O3a.

Where the readings of a pipeline detail give different outcomes, the outcome is taken under the
reading the paper best supports, adopted as an assumption (protocol, author step 1.6), and every
reading is shown. This applies to the claims of absence too (irc-tom-behavior-age, irc-fb-across,
maturity-within, rc-fb-no-difference): the claims file names one test for each, but not the
pipeline readings, so the assumed readings decide (O2a and O3a for the inter-region measures, O7a
for event magnitudes), and the outcome is marked as resting on them.


In [ ]:
READINGS = [(v, s, m) for v in ["all", "with_data"] for s in [False, True] for m in [False, True]]
def reading_name(r):
    return f"O1 {r[0]}, O2{'b' if r[1] else 'a'}, O3{'b' if r[2] else 'a'}"
MAIN = ("all", False, False)
TP = sb.TOM_ROIS + sb.PAIN_ROIS
BILATERAL = {"a": [("RTPJ", "LTPJ"), ("RS2", "LS2"), ("RInsula", "LInsula"), ("RMFG", "LMFG")],
             "b": [("RTPJ", "LTPJ"), ("RInsula", "LInsula")]}

def irc_measures(reading, m1):
    voxels, single, movie = reading
    trs = np.arange(sb.FIRST_MOVIE_TR - 1, sb.N_TR) if movie else None
    rois = TP if m1 else sb.ALL_ROIS
    rows, mats = {}, {}
    for s in subs:
        tc = sb.subject_timecourses(D[s], rois, voxels, m1, single)
        m = sb.irc_matrix(tc, rois, trs)
        mats[s] = m
        row = {"wToM": sb.within(m, sb.TOM_ROIS), "wPain": sb.within(m, sb.PAIN_ROIS),
               "across": sb.across(m, sb.TOM_ROIS, sb.PAIN_ROIS)}
        for k, pairs in BILATERAL.items():
            row[f"wToM_nonbi_{k}"] = sb.within(m, sb.TOM_ROIS, pairs)
            row[f"wPain_nonbi_{k}"] = sb.within(m, sb.PAIN_ROIS, pairs)
        if not m1:
            for net in ["Face", "Scene", "M1", "V1"]:
                row["w" + net] = sb.within(m, sb.NETWORKS[net])
            row["aFaceScene"] = sb.across(m, sb.FACE_ROIS, sb.SCENE_ROIS)
            row["aM1V1"] = sb.across(m, sb.M1_ROIS, sb.V1_ROIS)
        rows[s] = row
    return pd.DataFrame(rows).T.join(p), mats

IRC = {r: irc_measures(r, True) for r in READINGS}          # M1 regressed: main-text analyses
IRC_NOM1 = {r: irc_measures(r, False) for r in READINGS}    # no M1 regression: Supplementary Fig. 3
irc = IRC[MAIN][0]
print(irc.groupby("AgeGroup")[["wToM", "wPain", "across"]].agg(["mean", "sem"]).round(3).to_string())
print("ranges:", irc[["wToM", "wPain", "across"]].agg(["min", "max"]).round(3).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
order = sb.AGE_GROUPS + ["Adult"]
for ax, v in zip(axes, ["wToM", "wPain", "across"]):
    ax.boxplot([irc.loc[irc.AgeGroup == g, v] for g in order], tick_labels=order)
    ax.axhline(0, color="grey", lw=0.5); ax.set_title(v)
axes[0].set_ylabel("Fisher z (main reading)")
plt.tight_layout(); plt.show()
print("all values shown; y range", round(irc[["wToM", "wPain", "across"]].min().min(), 3), "to", round(irc[["wToM", "wPain", "across"]].max().max(), 3))

### irc-adult

> In adults, within-ToM (mean z .48) and within-Pain (.35) correlations are strongly positive and
> the across-network correlation is negative (-.17), each within-network value exceeding the
> across-network value.

In [ ]:
def tol(v, e, t):
    return abs(v - e) <= t + 1e-12
rows = []
for r in READINGS:
    a = IRC[r][0].loc[adults.index]
    t1 = stats.ttest_rel(a.wToM, a.across); t2 = stats.ttest_rel(a.wPain, a.across)
    rows.append({"reading": reading_name(r), "wToM": a.wToM.mean(), "wPain": a.wPain.mean(), "across": a.across.mean(),
                 "t ToM-across": t1.statistic, "p1": t1.pvalue, "t Pain-across": t2.statistic, "p2": t2.pvalue})
res = pd.DataFrame(rows)
res["values in tolerance"] = [sum([tol(x.wToM, .48, .02), tol(x.wPain, .35, .02), tol(x.across, -.17, .02),
                                    tol(x["t ToM-across"], 19.1, 1.91), tol(x["t Pain-across"], 23.2, 2.32)]) for _, x in res.iterrows()]
print(res.to_string(float_format=lambda x: f"{x:.3g}"))
print("expected: .48, .35, -.17, t 19.1, t 23.2 (5 values)")

In [ ]:
# Variants that are not the paper's method, to show how far the adult and 3yo means move with the
# filter and the CompCor regressors (main reading otherwise).
def means_with(clean_fn, idx):
    vals = []
    for s in idx:
        tc = {r: clean_fn(s, r) for r in TP}
        m = sb.irc_matrix(tc, TP)
        vals.append([sb.within(m, sb.TOM_ROIS), sb.within(m, sb.PAIN_ROIS), sb.across(m, sb.TOM_ROIS, sb.PAIN_ROIS)])
    return np.round(np.mean(vals, axis=0), 3)
def m1_of(s):
    r, l = D[s]["rois"]["right_primotor"], D[s]["rois"]["left_primotor"]
    return (r["all"] * r["n_vox"] + l["all"] * l["n_vox"]) / (r["n_vox"] + l["n_vox"])
def variant(hp_cutoff=sb.HIGHPASS_S, compcor=True):
    def f(s, r):
        d = D[s]; bad = sb.artifact_index(d["artifacts"])
        y = sb.nearest_neighbour_fill(d["rois"][r]["all"], bad) if len(bad) else d["rois"][r]["all"].copy()
        parts = ([d["compcor"].T] if compcor else []) + ([d["artifacts"].T] if len(bad) else []) + [m1_of(s)[:, None]]
        y = sb.residualize(y, np.column_stack(parts))
        if hp_cutoff:
            hp = sb.dct_highpass_basis(sb.N_TR, sb.TR_S, hp_cutoff); y = y - hp @ np.linalg.lstsq(hp, y, rcond=None)[0]
        y[bad] = np.nan
        return y
    return f
y3_idx = children[children.AgeGroup == "3yo"].index
for name, fn in {"paper (100 s, CompCor)": variant(), "no high-pass": variant(None), "50 s high-pass": variant(50.0), "no CompCor": variant(compcor=False)}.items():
    print(f"{name:24s} adults (wToM, wPain, across) {means_with(fn, adults.index)}   3yo {means_with(fn, y3_idx)}")
print("paper: adults .48, .35, -.17; 3yo .21, .23, .05")

In [ ]:
# Where the offset from the paper lies. The mean 13 x 13 matrices (main reading) of adults and
# 3-year-olds; Fig. 2 shows these matrices as colours only, with no numbers, so a pair-by-pair
# comparison with the paper is not possible.
for name, idx in [("adults", adults.index), ("3yo", y3_idx)]:
    M = pd.DataFrame(np.mean([IRC[MAIN][1][s].values for s in idx], axis=0), index=TP, columns=TP)
    print(f"mean Fisher z, {name} (n = {len(idx)}), main reading:")
    print(M.round(2).to_string())
    print()
# Diagnostic variants, none of them the paper's stated method: averaging r in place of Fisher z;
# the high-pass basis in the same regression as the nuisance covariates (as in an SPM GLM); left
# and right M1 as two regressors; and, to test whether a shared signal accounts for the offset,
# the mean ToM and Pain signal added as a regressor.
def diag_variant(kind):
    def f(s, r):
        d = D[s]; bad = sb.artifact_index(d["artifacts"])
        y = sb.nearest_neighbour_fill(d["rois"][r]["all"], bad) if len(bad) else d["rois"][r]["all"].copy()
        art = [d["artifacts"].T] if len(bad) else []
        m1 = np.column_stack([d["rois"][k]["all"] for k in sb.M1_ROIS]) if kind == "M1 left and right" else m1_of(s)[:, None]
        extra = [m1]
        if kind == "plus mean ToM and Pain signal":
            extra.append(np.mean([d["rois"][k]["all"] for k in TP], axis=0)[:, None])
        hp = sb.dct_highpass_basis()
        if kind == "high-pass in the nuisance regression":
            y = sb.residualize(y, np.column_stack([d["compcor"].T] + art + extra + [hp]))
        else:
            y = sb.residualize(y, np.column_stack([d["compcor"].T] + art + extra))
            y = y - hp @ np.linalg.lstsq(hp, y, rcond=None)[0]
        y[bad] = np.nan
        return y
    return f
def mean_r(idx):
    v = [[sb.within(np.tanh(IRC[MAIN][1][s]), sb.TOM_ROIS), sb.within(np.tanh(IRC[MAIN][1][s]), sb.PAIN_ROIS),
          sb.across(np.tanh(IRC[MAIN][1][s]), sb.TOM_ROIS, sb.PAIN_ROIS)] for s in idx]
    return np.round(np.mean(v, axis=0), 3)
print(f"{'mean of r, not z':40s} adults (wToM, wPain, across) {mean_r(adults.index)}   3yo {mean_r(y3_idx)}")
for kind in ["high-pass in the nuisance regression", "M1 left and right", "plus mean ToM and Pain signal"]:
    fn = diag_variant(kind)
    print(f"{kind:40s} adults (wToM, wPain, across) {means_with(fn, adults.index)}   3yo {means_with(fn, y3_idx)}")
print("paper: adults .48, .35, -.17; 3yo .21, .23, .05")


**Result: partially reproduced.** Within-ToM and within-Pain correlations are strongly positive in adults and above the across-network value (t = 19.1 and 21.8 under the main reading, both p < 1e-18), but the means are .604, .398 and -.051 against .48, .35 and -.17, all three out of tolerance under every reading; the two t values are within tolerance under three of the four O2 and O3 readings. The across-network mean is negative (-.051) but much weaker than the paper's. None of the variants outside the paper's method (no high-pass, 50 s high-pass, no CompCor) brings the means to the paper's. The adult and 3yo mean matrices are printed above; Fig. 2 gives them only as colours, so the offset cannot be traced pair by pair. The masks are in MNI mm (centroids of the 9 mm spheres within 3.7 mm of the stated centers and every voxel within 8.9 mm; M1 and V1 centered exactly, radius 10 mm), and the images are int16 with no scaling (maximum value 931 to 4081 across participants). Of the diagnostic variants, averaging r in place of z gives adults .519, .354, -.049, the high-pass in the nuisance regression .593, .393, -.086, and left and right M1 as two regressors .599, .390, -.037; only adding the mean ToM and Pain signal as a regressor moves the across-network value past the paper's (.286, .258, -.382). A shared signal that the paper's processing removed and this one does not would give an offset in this direction, but no step of the Methods' pipeline accounts for it.


### irc-age-within and irc-age-across

> Among children, within-ToM and within-Pain correlations increase with age, controlling for
> motion. Among children, the across-network correlation decreases with age.

Partial Spearman correlation with the number of artifact timepoints as covariate (ppcor form).

In [ ]:
rows = []
for r in READINGS:
    c = IRC[r][0].loc[children.index]
    row = {"reading": reading_name(r)}
    for v in ["wToM", "wPain", "across"]:
        rr, pv, df = sb.partial_spearman(c[v], c.Age, c.n_art)
        row[v + " rs"], row[v + " p"] = rr, pv
    rows.append(row)
age_irc = pd.DataFrame(rows)
print(age_irc.to_string(float_format=lambda x: f"{x:.3g}"))
print("df =", df, "; expected wToM .37 (p < .00005), wPain .28 (p = .002), across -.35 (p < .0001), tolerance 10%")

**Result: not reproduced (irc-age-within), not reproduced (irc-age-across), by Rule S.** Within-ToM rises with age (partial rs = .306, p = .00064) and within-Pain too (.461, p = 1.1e-07), and the across-network correlation falls (-.259, p = .0041), so the directions hold and all three are significant at .05. But rs within-ToM and across are out of tolerance (.37 and -.35, 10%) and their p-values do not reach the paper's bounds (p < .00005 and p < .0001) under any reading; within-Pain is much stronger than the paper's .28. Rule S counts a test that does not meet the paper's bound as failing.

### irc-adult-vs-child

> Adults have higher within-ToM and within-Pain correlations and a more negative across-network
> correlation than children, controlling for motion.

Regression of each measure on group (child = 1, adult = 0) and number of artifact timepoints.

In [ ]:
rows = []
for r in READINGS:
    t = IRC[r][0].copy(); t["child"] = (t.Child_Adult == "child").astype(float)
    for v in ["wToM", "wPain", "across"]:
        f = sb.ols(t, v, ["child", "n_art"]); fz = sb.ols(t, v, ["child", "n_art"], zscore=True)
        rows.append({"reading": reading_name(r), "measure": v, "b raw": f.params["child"], "b z (O4a)": fz.params["child"],
                     "t": f.tvalues["child"], "p": f.pvalues["child"], "t motion": f.tvalues["n_art"], "p motion": f.pvalues["n_art"]})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("expected t: wToM -5.7, wPain -3.8, across 7.2 (10%); paper motion effect: wToM t = -4.3, others n.s.")

**Result: partially reproduced.** Adults have higher within-ToM (t = -6.54 for the child effect, p = 8.7e-10) and within-Pain (t = -6.72, p = 3.5e-10) correlations and a more negative across-network correlation (t = 4.49, p = 1.4e-05) than children, so the tests hold with the paper's signs, but the t values are out of tolerance (10% of -5.7, -3.8 and 7.2): within-Pain (-6.48 to -6.72) and across (3.92 to 4.49) under every reading, and within-ToM under three of the four O2 and O3 readings (-6.54, -6.71, -7.05; -6.23 under O2a with O3b is inside). z-scaled b values are -1.16, -1.20 and .83 (paper -.97, -.75, 1.26). The motion effects differ from the paper's: on within-ToM t = -1.64 (paper -4.3), on across t = 2.87, p = .005 (paper not significant).

### irc-control-networks and irc-williams

> Without M1 regression, within-ToM (rs .39), within-Pain (.38), within-Scene (.23) and within-M1
> (.30) correlations rise with age in children, within-Face and within-V1 do not, and the across
> ToM-Pain correlation falls with age (-.26) while across Face-Scene and M1-V1 do not change.

The face and scene parcels are those deposited in derivatives/ROIs (the same Julian et al. 2012
parcels the claims file names as an outside input).

In [ ]:
CTRL = ["wToM", "wPain", "wScene", "wM1", "across", "wFace", "wV1", "aFaceScene", "aM1V1"]
EXP = {"wToM": .39, "wPain": .38, "wScene": .23, "wM1": .30, "across": -.26}
rows = []
for r in READINGS:
    c = IRC_NOM1[r][0].loc[children.index]
    for v in CTRL:
        rr, pv, _ = sb.partial_spearman(c[v], c.Age, c.n_art)
        rows.append({"reading": reading_name(r), "measure": v, "rs": rr, "p": pv,
                     "expected": EXP.get(v, np.nan), "in tolerance": tol(rr, EXP[v], abs(EXP[v]) * .1) if v in EXP else None})
ctrl = pd.DataFrame(rows)
print(ctrl.pivot(index="reading", columns="measure", values="rs")[CTRL].round(3).to_string())
print(ctrl.pivot(index="reading", columns="measure", values="p")[CTRL].map(lambda x: f"{x:.2g}").to_string())
print("paper: wToM .39 (<.0001), wPain .38 (<.0001), wScene .23 (.01), wM1 .30 (.001), across -.26 (.005); wFace .06 (.53), wV1 -.16 (.09), aFaceScene .02 (.8), aM1V1 .18 (.05)")
print("adult means (main reading):", IRC_NOM1[MAIN][0].loc[adults.index, CTRL].mean().round(3).to_dict())

**Result: partially reproduced.** The ToM and Pain age effects hold (within-ToM rs = .367, p = 3.5e-05; within-Pain .525, p = 6.4e-10; across -.247, p = .0064), with within-ToM and across in tolerance and within-Pain out (.38 expected). Within-Scene (.288, p = .0014) and within-M1 (.227, p = .012) rise with age as in the paper, both out of tolerance. Within-Face (.106, p = .25), within-V1 (-.069, p = .45) and across Face-Scene (.065, p = .48) show no age effect, as in the paper, but across M1-V1 does (.200, p = .028; paper .18, p = .05) under every reading.

In [ ]:
PAIRS = [("wToM", "wFace"), ("wPain", "wFace"), ("wToM", "wScene"), ("wPain", "wScene"), ("wToM", "wM1"), ("wPain", "wM1"),
         ("wToM", "wV1"), ("wPain", "wV1"), ("across", "aFaceScene"), ("across", "aM1V1")]
SHOULD = [True, True, False, False, False, False, True, True, True, True]
rows = []
for r in READINGS:
    c = IRC_NOM1[r][0].loc[children.index]
    for reading_w in ["a (partial, motion)", "b (no covariate)"]:
        for (x, y), should in zip(PAIRS, SHOULD):
            if reading_w.startswith("a"):
                rx = sb.partial_spearman(c[x], c.Age, c.n_art)[0]; ry = sb.partial_spearman(c[y], c.Age, c.n_art)[0]
                rxy = sb.partial_spearman(c[x], c[y], c.n_art)[0]
            else:
                rx = stats.spearmanr(c[x], c.Age).statistic; ry = stats.spearmanr(c[y], c.Age).statistic
                rxy = stats.spearmanr(c[x], c[y]).statistic
            w = sb.williams(rx, ry, rxy, len(c))
            sig = w["p_t"] < .05
            same_dir = (abs(rx) > abs(ry))
            rows.append({"reading": reading_name(r), "williams": reading_w, "pair": f"{x} vs {y}", "r age x": rx, "r age y": ry,
                         "r xy": rxy, "t": w["t"], "p_t": w["p_t"], "z": w["z"], "p_z": w["p_z"], "paper significant": should,
                         "as paper": (sig and same_dir) if should else (not sig)})
wil = pd.DataFrame(rows)
print(wil[(wil.reading == reading_name(MAIN))].drop(columns="reading").to_string(float_format=lambda x: f"{x:.3g}"))
print(wil.groupby(["reading", "williams"])["as paper"].sum().rename("comparisons as in the paper, of 10").to_string())

**Result: not reproduced.** The six differences the paper finds significant are significant in the same direction under both readings (Face and V1 against ToM and Pain, t = 2.51 to 5.03; across ToM-Pain against Face-Scene and M1-V1, t = -2.95 and -4.0, main reading, reading a). Of the four the paper finds not significant, ToM against Scene (t = .75) and ToM against M1 (t = 1.32) are not, but Pain against Scene (t = 2.62, p = .0098) and Pain against M1 (t = 2.92, p = .0043) are, so the within-Pain age effect is stronger than the Scene and M1 effects, unlike the paper. 8 of 10 comparisons follow the paper under every reading. The criterion requires all ten and states no partial outcome, and two of the paper's tests (p = .19 and .47) come out on the other side of .05, which Rule S counts as failed tests.


### irc-3yo-distinct, irc-3yo-not-anticorrelated, irc-3yo-homologous, irc-3yo-nonbilateral

The 17 children of the 3yo group.

In [ ]:
y3 = children[children.AgeGroup == "3yo"].index
rows = []
for r in READINGS:
    t = IRC[r][0].loc[y3]
    t1 = stats.ttest_rel(t.wToM, t.across); t2 = stats.ttest_rel(t.wPain, t.across)
    rows.append({"reading": reading_name(r), "wToM": t.wToM.mean(), "sem ToM": t.wToM.sem(), "wPain": t.wPain.mean(),
                 "sem Pain": t.wPain.sem(), "t ToM-across": t1.statistic, "p1": t1.pvalue, "t Pain-across": t2.statistic, "p2": t2.pvalue,
                 "in tolerance (of 4)": sum([tol(t.wToM.mean(), .21, .02), tol(t.wPain.mean(), .23, .02),
                                            tol(t1.statistic, 6.2, .62), tol(t2.statistic, 6.9, .69)])})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("expected wToM .21, wPain .23 (tolerance .02), t 6.2 and 6.9 (10%), n = 17")

**Result: not reproduced, by Rule S.** In the 17 three-year-olds within-ToM (.326) and within-Pain (.179) exceed the across-network value (t = 8.43, p = 2.8e-07; t = 4.54, p = .00034), so the networks are distinct, but none of the four values is within tolerance (.21, .23, 6.2, 6.9), and the Pain test does not reach the paper's bound of p < .00001 under any reading.

In [ ]:
rows = []
for r in READINGS:
    a = IRC[r][0].loc[y3, "across"]
    tt = stats.ttest_1samp(a, 0)
    rows.append({"reading": reading_name(r), "mean across": a.mean(), "sem": a.sem(), "t vs 0 (variant)": tt.statistic, "p": tt.pvalue})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("expected .05 (tolerance .02); adults (main reading)", round(IRC[MAIN][0].loc[adults.index, "across"].mean(), 3))

**Result: partially reproduced.** The 3yo across-network mean is positive, .074 (s.e. .020), not anti-correlated, against adults' -.051; it is just outside the tolerance (.05 ± .02) under every reading (.074 to .075). A one-sample t-test shows it above 0 (t = 3.75, p = .0017).

In [ ]:
for r in READINGS:
    M = pd.DataFrame(np.mean([IRC[r][1][s].values for s in y3], axis=0), index=TP, columns=TP)
    print(reading_name(r))
    for net, rois, homol in [("ToM", sb.TOM_ROIS, ("RTPJ", "LTPJ")), ("Pain", sb.PAIN_ROIS, ("RInsula", "LInsula"))]:
        pairs = sorted([(M.loc[a, b], a, b) for i, a in enumerate(rois) for b in rois[i + 1:]], reverse=True)
        rank = [i for i, (v, a, b) in enumerate(pairs) if {a, b} == set(homol)][0] + 1
        print(f"  {net} pairs by 3yo mean z:", ", ".join(f"{a}-{b} {v:.3f}" for v, a, b in pairs[:5]),
              f"| {homol[0]}-{homol[1]} ranks {rank} of {len(pairs)}")
M = pd.DataFrame(np.mean([IRC[MAIN][1][s].values for s in y3], axis=0), index=TP, columns=TP)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(M.values, vmin=-0.6, vmax=0.6, cmap="RdBu_r"); ax.set_xticks(range(13), TP, rotation=90); ax.set_yticks(range(13), TP)
plt.colorbar(im, label="mean Fisher z, 3yo"); plt.tight_layout(); plt.show()
print("matrix range (off-diagonal):", round(np.nanmin(M.values), 3), "to", round(np.nanmax(M.values), 3), "; colour scale -0.6 to 0.6")


**Result: not reproduced.** In the 3yo mean matrix the strongest Pain pair is RInsula-LInsula (.856), and the three homologous Pain pairs are the top three, but the strongest ToM pair is RTPJ-PC (.637); RTPJ-LTPJ (.558) ranks fourth of 15. Neither reading of the criterion holds for ToM.

In [ ]:
rows = []
for r in READINGS:
    t = IRC[r][0].loc[y3]
    for k in ["a", "b"]:
        t1 = stats.ttest_rel(t[f"wToM_nonbi_{k}"], t.across); t2 = stats.ttest_rel(t[f"wPain_nonbi_{k}"], t.across)
        rows.append({"reading": reading_name(r), "pairs dropped": k, "ToM": t[f"wToM_nonbi_{k}"].mean(), "Pain": t[f"wPain_nonbi_{k}"].mean(),
                     "t ToM": t1.statistic, "p ToM": t1.pvalue, "t Pain": t2.statistic, "p Pain": t2.pvalue,
                     "in tolerance (of 4)": sum([tol(t[f"wToM_nonbi_{k}"].mean(), .20, .02), tol(t[f"wPain_nonbi_{k}"].mean(), .17, .02),
                                                tol(t1.statistic, 5.1, .51), tol(t2.statistic, 4.4, .44)])})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("expected ToM .20, Pain .17 (tolerance .02); t 5.1 (p = .0001), 4.4 (p = .0005)")

**Result: partially reproduced.** Under reading (b), dropping only the two pairs the paper names, non-bilateral within-ToM (.309, t = 7.64, p = 1e-06) and within-Pain (.146, t = 3.03, p = .008) exceed the across-network value, so the tests hold, but the means and t values are out of tolerance (except the Pain mean under O2b, .152 and .153). Under reading (a), dropping all three Pain pairs, non-bilateral Pain (.106) does not differ from across (t = 1.34, p = .20).

### irc-tom-behavior and irc-tom-behavior-age

In [ ]:
rows = []
for r in READINGS:
    c = IRC[r][0].loc[children.index]
    a = sb.partial_kendall(c.wToM, c[TOM], c.n_art); b = sb.partial_kendall(c.across, c[TOM], c.n_art)
    rows.append({"reading": reading_name(r), "tau wToM": a[0], "p wToM": a[1], "tau across": b[0], "p across": b[1]})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("expected wToM .23 (p = .0002), across -.20 (p = .001), tolerance 10%")

**Result: partially reproduced.** Within-ToM correlation rises with ToM score controlling for motion (partial tau = .246, p = 6.3e-05; paper .23, within tolerance). The across-network correlation falls with ToM score (tau = -.125, p = .042) but less than the paper's -.20, out of tolerance; both tests are significant with the paper's signs.

In [ ]:
rows = []
for r in READINGS:
    c = IRC[r][0].loc[children.index]
    for v in ["wToM", "across"]:
        f = sb.ols(c, v, [TOM, "Age", "n_art"]); fz = sb.ols(c, v, [TOM, "Age", "n_art"], zscore=True)
        rows.append({"reading": reading_name(r), "measure": v, "b ToM raw": f.params[TOM], "b ToM z": fz.params[TOM],
                     "t ToM": f.tvalues[TOM], "p ToM": f.pvalues[TOM], "t age": f.tvalues["Age"], "p age": f.pvalues["Age"]})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("paper: ToM score ts < 1, ps > .3")

**Result: not reproduced, resting on an assumption.** Under the main reading (O2a, O3a) the ToM score effect on within-ToM is significant with age and motion in the model (t = 2.04, p = .044); on across-network it is not (t = 1.29, p = .20). The readings disagree: O3b gives p = .069, O2b .048, O2b with O3b .088. The main reading is adopted (see deviations).

### irc-fb-within, irc-fb-within-dccs, irc-fb-across

Passers (30) against failers (15), fail coded 1, with age and motion (and DCCS).

In [ ]:
pf_idx = young[young.FB_Group.isin(["pass", "fail"])].index
rows = []
for r in READINGS:
    t = IRC[r][0].loc[pf_idx].copy(); t["fail"] = (t.FB_Group == "fail").astype(float)
    for v in ["wToM", "across"]:
        f = sb.ols(t, v, ["fail", "Age", "n_art"]); fz = sb.ols(t, v, ["fail", "Age", "n_art"], zscore=True)
        g = sb.ols(t, v, ["fail", "Age", "n_art", DCCS])
        rows.append({"reading": reading_name(r), "measure": v, "mean pass": t.loc[t.fail == 0, v].mean(), "mean fail": t.loc[t.fail == 1, v].mean(),
                     "b raw": f.params["fail"], "b z": fz.params["fail"], "t": f.tvalues["fail"], "p": f.pvalues["fail"],
                     "t with DCCS": g.tvalues["fail"], "p with DCCS": g.pvalues["fail"], "t DCCS": g.tvalues[DCCS], "p DCCS": g.pvalues[DCCS]})
fbirc = pd.DataFrame(rows)
print(fbirc.to_string(float_format=lambda x: f"{x:.3g}"))
print("paper: wToM pass .29, fail .25, b = -.70, t = -2.06, p = .046 (pass higher); with DCCS t = -1.80, p = .079; across .04 against .03, t = 1.2, p = .23")

**Result: not reproduced (irc-fb-within, irc-fb-within-dccs, irc-fb-across; irc-fb-across rests on an assumption).** Passers have a slightly higher within-ToM correlation than failers (.398 against .345), but with age and motion the group effect is not significant (t = .66, p = .51; paper p = .046) under any reading, and with DCCS added p = .69, outside the .05 to .10 band. Across-network, passers (.087) are above failers (.057) and the group effect is significant under the main reading (t = -2.18, p = .035), where the paper finds none; O3b gives p = .051, O2b .034, O2b with O3b .047.

## Reverse correlation

Network timecourses (P6) from ROI timecourses cleaned without the M1 regressor. O6: z over
TRs 11 to 168 (a) or all 168 (b). O5: p < .05 (a), p < .01 (b), FDR p < .05 over the 158
timepoints (c). O1 is fixed at the main reading (z-normalization makes O1's voxel choice
irrelevant up to the few partial voxels; shown in the IRC section).

In [ ]:
def network_tcs(net, z=True, z_span="movie", voxels="all"):
    rois = sb.FFA_ROIS if net == "FFA" else sb.NETWORKS[net]
    out = {}
    for s in subs:
        tc = sb.subject_timecourses(D[s], rois, voxels, False)
        out[s] = sb.network_timecourse(tc, rois, z_span, z)
    return pd.DataFrame(out).T

NETS = ["ToM", "Pain", "Face", "Scene", "M1", "V1", "FFA"]
NZ = {(n, span): network_tcs(n, True, span) for n in ["ToM", "Pain", "FFA"] for span in ["movie", "all"]}
RAW = {n: network_tcs(n, False) for n in NETS}
O5 = {"a p<.05": (0.05, False), "b p<.01": (0.01, False), "c FDR<.05": (0.05, True)}
RC = {}
for span in ["movie", "all"]:
    for k, (alpha, fdr) in O5.items():
        for n in ["ToM", "Pain", "FFA"]:
            tab, ev = sb.reverse_correlation(NZ[(n, span)].loc[adults.index].values, alpha, fdr)
            RC[(n, span, k)] = (tab, ev, sb.rank_events(ev, tab))
rows = []
for (n, span, k), (tab, ev, rk) in RC.items():
    if n == "FFA":
        continue
    m = sb.match_table_events(ev, n)
    rows.append({"network": n, "O6 z span": span, "O5": k, "events": len(ev), "seconds": 2 * sum(len(e) for e in ev),
                 "table events found": int(m.sum()), "of": len(m), "missed": ",".join(m.index[~m])})
rcsum = pd.DataFrame(rows)
print(rcsum.to_string())

In [ ]:
# Timing check (2): adult network contrast during the paper's ToM against pain events, with the
# event labels shifted in time.
tom_trs = np.array(sorted(tr_set_ := set(t for v in sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == "ToM"].trs for t in v)))
pain_trs = np.array(sorted(set(t for v in sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == "Pain"].trs for t in v)))
for shift in [-10, -4, -2, 0, 2, 4, 10]:
    out = []
    for n in ["ToM", "Pain"]:
        m = np.nanmean(NZ[(n, "movie")].loc[adults.index].values, axis=0)
        tt, pp = np.clip(tom_trs + shift, 11, 168) - 1, np.clip(pain_trs + shift, 11, 168) - 1
        out.append(m[tt].mean() - m[pp].mean())
    print(f"shift {shift:3d} TRs: ToM network, ToM minus pain events = {out[0]:.2f}; Pain network, ToM minus pain events = {out[1]:.2f}")

### rc-events

> Reverse correlation on the adult network timecourses finds seven ToM events (68 s, mean length
> 9.7 s) and twelve pain events (86 s, mean length 7.2 s), at the times of Supplementary Table 3.

In [ ]:
for (n, span, k), (tab, ev, rk) in RC.items():
    if n != "FFA" and span == "movie":
        print(n, k, "events (onset TR, TRs, peak TR):", [(int(a), int(b), int(c)) for a, b, c in zip(rk.onset, rk.length_tr, rk.peak_tr)])
        print("   mean length %.1f s (s.d. %.1f)" % (2 * rk.length_tr.mean(), 2 * rk.length_tr.std()) if len(rk) else "")
print(sb.SUPP_TABLE3[["event", "time", "duration_s", "peak_tr", "onset_tr"]].to_string(index=False))
score = rcsum.groupby(["O6 z span", "O5"]).apply(lambda g: pd.Series({
    "table events found": g["table events found"].sum(), "count error": (g.events - g["of"]).abs().sum(),
    "ToM events": int(g.loc[g.network == "ToM", "events"].iloc[0]), "Pain events": int(g.loc[g.network == "Pain", "events"].iloc[0]),
    "ToM s": int(g.loc[g.network == "ToM", "seconds"].iloc[0]), "Pain s": int(g.loc[g.network == "Pain", "seconds"].iloc[0])}), include_groups=False)
print(score.to_string())
# Most table events found, then smallest count error; ties between the two O6 spans go to O6b
# (all 168 TRs): the Methods (paragraph 9) z-normalize each ROI timecourse and only then leave out
# the first 10 timepoints from the t-tests.
score["O6 preference"] = [0 if span == "all" else 1 for span, _ in score.index]
best = score.sort_values(["table events found", "count error", "O6 preference"], ascending=[False, True, True]).index[0]
BEST_SPAN, BEST_O5 = best
print("best-matching reading (most table events found, then smallest count error, then O6b):", best)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, n, col in zip(axes, ["ToM", "Pain"], ["tab:red", "tab:green"]):
    tab, ev, rk = RC[(n, BEST_SPAN, BEST_O5)]
    ax.plot(tab.TR, tab["mean"], color=col)
    for e in ev:
        ax.axvspan(e[0] - 0.5, e[-1] + 0.5, color=col, alpha=0.2)
    for _, row in sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == n].iterrows():
        ax.axvspan(row.trs[0] - 0.5, row.trs[-1] + 0.5, fill=False, edgecolor="k", lw=0.8)
    ax.set_ylabel(f"adult mean, {n} (z)"); ax.axhline(0, color="grey", lw=0.5)
axes[1].set_xlabel("TR (shaded: found events; outlined: Supplementary Table 3)")
plt.tight_layout(); plt.show()
print("y ranges:", {n: (round(RC[(n, BEST_SPAN, BEST_O5)][0]["mean"].min(), 2), round(RC[(n, BEST_SPAN, BEST_O5)][0]["mean"].max(), 2)) for n in ["ToM", "Pain"]})

**Result: partially reproduced.** At p < .05 uncorrected (reading O5a, the best match to Supplementary Table 3; O6 makes no difference for these networks), all seven ToM events and 10 of 12 pain events of Supplementary Table 3 are found (P08 and P10 missed), but the counts are 9 ToM events (80 s) and 10 pain events (84 s), against 7 (68 s) and 12 (86 s). The ToM count matches under O5b and O5c (7 events, 56 s), where only 8 of 12 pain events are found. The pain time (84 s) is within tolerance; the ToM time is not under any reading.

### rc-overlap

> ToM and pain events overlap at a single timepoint (the last of T05 and first of P05), less
> overlap than chance in a permutation test.

Permutations as Methods paragraph 10, 1000 per reading, seed fixed for this section.

In [ ]:
rng = np.random.default_rng(20180301)
# Reading (b): 159-TR timecourses, the length implied by the paper's zeros (125 and 116) and
# event totals (34 and 43 TRs); with the Supplementary Table 3 events it gives 125 and 116 zeros.
def overlap_test(ev_a, ev_b, zeros_a, zeros_b, n_perm=1000):
    obs = len(set(t for e in ev_a for t in e) & set(t for e in ev_b for t in e))
    perm = sb.overlap_permutation([len(e) for e in ev_a], [len(e) for e in ev_b], zeros_a, zeros_b, n_perm, rng)
    return obs, (perm <= obs).sum(), perm.mean()
rows = []
evsets = {"found (" + BEST_SPAN + ", " + BEST_O5 + ")": (RC[("ToM", BEST_SPAN, BEST_O5)][1], RC[("Pain", BEST_SPAN, BEST_O5)][1])}
for k in O5:
    evsets[f"found (movie, {k})"] = (RC[("ToM", "movie", k)][1], RC[("Pain", "movie", k)][1])
evsets["Supplementary Table 3 events (variant)"] = (list(sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == "ToM"].trs), list(sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == "Pain"].trs))
for name, (et, ep) in evsets.items():
    nt, np_ = sum(map(len, et)), sum(map(len, ep))
    for reading, (za, zb) in {"(a) 158 TRs": (158 - nt, 158 - np_), "(b) 159 TRs": (159 - nt, 159 - np_)}.items():
        obs, k, mean = overlap_test(et, ep, za, zb)
        rows.append({"events": name, "reading": reading, "observed overlap TRs": obs, "perms <= observed": k, "p": k / 1000, "mean perm overlap": mean})
ovl = pd.DataFrame(rows)
print(ovl.to_string())
both = set(t for e in RC[("ToM", BEST_SPAN, BEST_O5)][1] for t in e) & set(t for e in RC[("Pain", BEST_SPAN, BEST_O5)][1] for t in e)
print("overlapping TRs of found events:", sorted(both))

**Result: partially reproduced.** The ToM and pain events found (O5a) do not overlap at all (0 TRs, where the paper has 1), which is less than chance: 2 of 1000 permutations have as little overlap under reading (a), 158 TRs (p = .002, mean permuted overlap 11.9 TRs), and none under (b), 159 TRs (p < .001, mean 11.9). Under O5b and O5c the overlap is also 0, p = .019 to .049. The overlap count differs from the paper's by one timepoint. The criterion states only when the claim is reproduced. The outcome is partial because the claim's test (less overlap than chance, p <= .01) holds and only its expected count is out of tolerance, which is what the claims file's Rule S and the protocol call a partial result; under irc-williams, by contrast, two of the paper's tests fail. With the paper's own events from Supplementary Table 3 the overlap is 1 TR (the last of T05 and first of P05), p = .002 (a) and .008 (b).


### rc-child-adult-timecourse and supp-table4-other-networks

Network timecourses before z-normalization, averaged over the participants of each age group
(artifact timepoints left out), correlated with the adult average over TRs 11 to 168.

In [ ]:
MOVIE = np.arange(sb.FIRST_MOVIE_TR - 1, sb.N_TR)
def group_r(net, idx):
    a = np.nanmean(RAW[net].loc[adults.index].values, axis=0)[MOVIE]
    c = np.nanmean(RAW[net].loc[idx].values, axis=0)[MOVIE]
    res = stats.pearsonr(a, c)
    return res.statistic, res.pvalue
rows = []
for net in NETS:
    for g in sb.AGE_GROUPS:
        r, pv = group_r(net, children[children.AgeGroup == g].index)
        rows.append({"network": net, "group": g, "r": r, "p": pv})
    r, pv = group_r(net, children.index)
    rows.append({"network": net, "group": "all children", "r": r, "p": pv})
gr = pd.DataFrame(rows)
print(gr.pivot(index="network", columns="group", values="r").round(3).to_string())
print(gr.pivot(index="network", columns="group", values="p").map(lambda x: f"{x:.1e}").to_string())

In [ ]:
EXP_TP = {"ToM": [.28, .31, .60, .72, .82], "Pain": [.60, .56, .73, .83, .89]}
# The paper's bounds (claims file statistics): ToM every bin p < .0005 (3yo exact p = .00046);
# Pain every bin p < 1e-13, 3yo p < 1e-15. Bonferroni alpha .01 for five bins.
BOUND = {"ToM": [.0005] * 5, "Pain": [1e-15, 1e-13, 1e-13, 1e-13, 1e-13]}
for net, e in EXP_TP.items():
    g = gr[(gr.network == net) & gr.group.isin(sb.AGE_GROUPS)].copy(); g["expected"] = e
    g["in tolerance"] = [tol(a, b, .1 * b) for a, b in zip(g.r, g.expected)]
    g["paper bound"] = BOUND[net]
    g["meets bound"] = g.p < g["paper bound"]
    g["p < .01"] = g.p < .01
    print(g.to_string(index=False, float_format=lambda x: f"{x:.3g}"))


**Result: not reproduced.** Every age group's mean timecourse correlates with the adult mean at p < .01 (the paper's Bonferroni alpha): ToM r = .212, .329, .564, .703, .785 and Pain .531, .516, .701, .797, .862. 8 of 10 values are within 10%; ToM 3yo (.212 against .28) and Pain 3yo (.531 against .60) are below. Three correlations miss the paper's bounds: ToM 3yo p = .0076 against p < .0005 (the paper's exact p is .00046), Pain 3yo p = 7.1e-13 against p < 1e-15, and Pain 4yo p = 4.0e-12 against p < 1e-13. The criterion's partial outcome needs every r significant at the paper's bounds, so it does not apply. Judged only at the Bonferroni alpha of .01, every r is significant and the claim would be partially reproduced.


In [ ]:
EXP_S4 = {"Face": [.75, .59, .71, .84, .86], "Scene": [.61, .67, .77, .80, .85], "M1": [.11, .06, .35, .44, .53],
          "V1": [.53, .61, .78, .74, .76], "FFA": [.72, .60, .68, .82, .83]}
rows = []
for net, e in EXP_S4.items():
    g = gr[(gr.network == net) & gr.group.isin(sb.AGE_GROUPS)]
    for (_, x), ex in zip(g.iterrows(), e):
        rows.append({"network": net, "group": x.group, "r": x.r, "expected": ex, "in tolerance": tol(x.r, ex, .1 * ex),
                     "p": x.p, "significant (p < .0005)": x.p < .0005})
s4 = pd.DataFrame(rows)
print(s4.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("in tolerance:", int(s4["in tolerance"].sum()), "of 25; not significant:", list(s4.loc[~s4["significant (p < .0005)"], ["network", "group"]].itertuples(index=False, name=None)))

**Result: not reproduced.** 17 of the 25 values are within 10% (the criterion's partial outcome needs 20). The significance pattern holds at p < .0005 (only 3yo M1, p = .00078, and 4yo M1, p = .019, are not), but the M1 values differ throughout (.265, .186, .434, .350, .315 against .11, .06, .35, .44, .53), and the 3yo Face (.637 against .75), V1 (.442 against .53) and FFA (.589 against .72) values are low.

### rc-age-events, rc-t04-tom, rc-fb-no-difference

Response magnitude at the peak TR of each Supplementary Table 3 event, ToM events from the ToM
network and pain events from the Pain network. O7a: z-normalized timecourse (O6 a or b); O7b:
before z-normalization. Children whose peak TR is an artifact timepoint (NaN) are left out of
that event's test; their number is printed.

In [ ]:
MAG = {}
for name, src in {"O7a z movie": lambda n: NZ[(n, "movie")], "O7a z all": lambda n: NZ[(n, "all")], "O7b raw": lambda n: RAW[n]}.items():
    MAG[name] = pd.DataFrame({row.event: src(row.network)[row.peak_tr - 1] for _, row in sb.SUPP_TABLE3.iterrows()})
rows = []
for name, mag in MAG.items():
    for ev in mag.columns:
        c = children.join(mag[[ev]]).dropna(subset=[ev])
        rr, pv, _ = sb.partial_spearman(c[ev], c.Age, c.n_art)
        rows.append({"reading": name, "event": ev, "n": len(c), "rs": rr, "p": pv, "passes p < .0026": pv < .0026 and rr > 0})
age_ev = pd.DataFrame(rows)
print(age_ev.pivot(index="event", columns="reading", values="rs").round(3).join(
      age_ev.pivot(index="event", columns="reading", values="p").map(lambda x: f"{x:.1e}"), rsuffix=" p").to_string())
print("children left out (artifact at peak):", age_ev.groupby("event").n.first().rsub(len(children)).to_dict())
for name in MAG:
    passed = list(age_ev[(age_ev.reading == name) & age_ev["passes p < .0026"]].event)
    print(name, "events passing:", passed, "; with rs > .28:", list(age_ev[(age_ev.reading == name) & age_ev["passes p < .0026"] & (age_ev.rs > .28)].event))
print("paper: P01, P04, P08, T01, T02")

**Result: not reproduced.** Under the z-normalized magnitudes (O7a, either O6 span), four events pass p < .0026 with positive rs: T02 (.325), T07 (.352), P04 (.324) and T03 (.274). Of the paper's five (P01, P04, P08, T01, T02), only P04 and T02 pass; P01 (rs = .193, p = .038), T01 (.270, p = .0036) and P08 (.019, p = .84) do not, and T03 and T07 pass where the paper has them not. Under O7b no event passes.

In [ ]:
rows = []
for name, mag in MAG.items():
    for ev in [e for e in mag.columns if e.startswith("T")]:
        c = children.join(mag[[ev]]).dropna(subset=[ev])
        f = sb.ols(c, ev, [TOM, "Age", "n_art"]); fz = sb.ols(c, ev, [TOM, "Age", "n_art"], zscore=True)
        rows.append({"reading": name, "event": ev, "n": int(f.nobs), "b raw": f.params[TOM], "b z": fz.params[TOM], "t": f.tvalues[TOM], "p": f.pvalues[TOM], "p < .007": f.pvalues[TOM] < .007})
t04 = pd.DataFrame(rows)
print(t04.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("paper T04: b = .4, t = 2.98, p = .0035; only T04 marked")

**Result: not reproduced.** The T04 magnitude does not rise with ToM score with age and motion in the model: t = 1.18, p = .24 (O7a, TRs 11 to 168; t = 1.14 and .71 under the other readings), against t = 2.98, p = .0035. No ToM event passes p < .007 under any reading.

In [ ]:
rng_fb = np.random.default_rng(1027)
rows = []
for name, mag in MAG.items():
    for ev in [e for e in mag.columns if e.startswith("T")]:
        c = young.loc[pf_idx].join(mag[[ev]]).dropna(subset=[ev]).copy(); c["fail"] = (c.FB_Group == "fail").astype(float)
        f = sb.ols(c, ev, ["fail", "Age", "n_art"])
        row = {"reading": name, "event": ev, "n": int(f.nobs), "mean pass": c.loc[c.fail == 0, ev].mean(), "mean fail": c.loc[c.fail == 1, ev].mean(),
               "t": f.tvalues["fail"], "p": f.pvalues["fail"]}
        if ev == "T03":
            t_obs = f.tvalues["fail"]; perm = []
            for _ in range(5000):
                cc = c.copy(); cc["fail"] = rng_fb.permutation(cc.fail.values)
                perm.append(sb.ols(cc, ev, ["fail", "Age", "n_art"]).tvalues["fail"])
            row["p permutation (T03)"] = float(np.mean(np.abs(perm) >= abs(t_obs)))
        rows.append(row)
fbev = pd.DataFrame(rows)
print(fbev.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("smallest p per reading:", fbev.groupby("reading").p.min().round(4).to_dict(), "(paper: all p > .08)")

**Result: reproduced, resting on an assumption.** Under the z-normalized magnitudes (O7a), no ToM event differs between passers and failers with age and motion in the model (smallest p = .091 for T05 with O6a, .072 with O6b; T03 permutation p = .72). Under O7b, magnitudes before z-normalization, T05 has p = .0497, just below .05. O7a is adopted (see deviations).

### rc-3yo-events

> Reverse correlation on the 3-year-olds alone finds 4 of the 12 adult pain events and 1 of the 7
> adult ToM events; of its 32 s of pain events 14 s fall in adult pain events and 8 s in adult ToM
> event T04, and of its 8 s of ToM events 4 s fall in an adult ToM event and 4 s in adult pain
> event P01.

Adult events are those of Supplementary Table 3 (Figure 4a); the events found here in adults
are shown as a variant.

In [ ]:
def tr_set(evs):
    return set(t for e in evs for t in e)
TAB = {n: dict(zip(sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == n].event, sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == n].trs)) for n in ["ToM", "Pain"]}
rows = []
for span in ["movie", "all"]:
    for k, (alpha, fdr) in O5.items():
        e3 = {n: sb.reverse_correlation(NZ[(n, span)].loc[y3].values, alpha, fdr)[1] for n in ["ToM", "Pain"]}
        row = {"O6": span, "O5": k}
        for n, other in [("Pain", "ToM"), ("ToM", "Pain")]:
            s3 = tr_set(e3[n])
            row[f"3yo {n} events"] = len(e3[n]); row[f"3yo {n} s"] = 2 * len(s3)
            row[f"adult {n} events found"] = sum(bool(s3 & set(v)) for v in TAB[n].values())
            row[f"{n} s in adult {n} events"] = 2 * len(s3 & tr_set(TAB[n].values()))
            row[f"{n} s in adult {other} events"] = 2 * len(s3 & tr_set(TAB[other].values()))
        row["Pain s in T04"] = 2 * len(tr_set(e3["Pain"]) & set(TAB["ToM"]["T04"]))
        row["ToM s in P01"] = 2 * len(tr_set(e3["ToM"]) & set(TAB["Pain"]["P01"]))
        row["3yo events (TR lists)"] = {n: [(e[0], len(e)) for e in e3[n]] for n in e3}
        row["vs found adult events: Pain s"] = 2 * len(tr_set(e3["Pain"]) & tr_set(RC[("Pain", span, k)][1]))
        row["vs found adult events: ToM s"] = 2 * len(tr_set(e3["ToM"]) & tr_set(RC[("ToM", span, k)][1]))
        rows.append(row)
r3 = pd.DataFrame(rows)
print(r3.drop(columns="3yo events (TR lists)").T.to_string())
for _, x in r3.iterrows():
    print(x.O6, x.O5, x["3yo events (TR lists)"])
print("paper: 3yo pain 32 s, 4 of 12 adult pain events, 14 s in adult pain events, 8 s in T04; 3yo ToM 8 s, 1 of 7 adult ToM events, 4 s in a ToM event, 4 s in P01")
print("reading of rc-events used by the criterion:", BEST_SPAN, BEST_O5)

**Result: partially reproduced, resting on an assumption.** No reading reproduced rc-events, so the criterion's "reading that reproduced rc-events" is extended to the reading that matched it best (O5a). Under O5a with O6b, the 3yo pain events (40 s) overlap 6 of the 12 adult pain events (22 s in adult pain events) and adult ToM event T04 (6 s), and the 3yo ToM events (24 s) overlap 1 of 7 adult ToM events (4 s) and adult pain event P01 (4 s). The two cross-network overlaps of the claim hold; the times and the pain-event count do not (paper 32 s, 4 of 12, 14 s, 8 s in T04; 8 s of ToM events). Under O6a the 3yo ToM events do not overlap P01 (0 s), so the claim would not be reproduced; O6b is adopted (see deviations).

## Functional maturity

Each child's ToM (Pain) network timecourse before z-normalization, TRs 11 to 168, artifact
timepoints left out, correlated (Pearson) with the adult mean timecourse (Methods paragraph 13).
IRC measures are those of irc-age-within (M1 regressed), under each IRC reading.

In [ ]:
def maturity(net):
    adult_mean = np.nanmean(RAW[net].loc[adults.index].values, axis=0)[MOVIE]
    out = {}
    for s in children.index:
        x = RAW[net].loc[s].values[MOVIE]; ok = np.isfinite(x)
        out[s] = stats.pearsonr(x[ok], adult_mean[ok]).statistic
    return pd.Series(out)
MAT = pd.DataFrame({"mat_ToM": maturity("ToM"), "mat_Pain": maturity("Pain")})
print(MAT.describe().round(3).to_string())

In [ ]:
rows = []
for r in READINGS:
    c = IRC[r][0].loc[children.index].join(MAT)
    for net, wv in [("ToM", "wToM"), ("Pain", "wPain")]:
        f = sb.ols(c, f"mat_{net}", ["across", wv, "Age", "n_art"]); fz = sb.ols(c, f"mat_{net}", ["across", wv, "Age", "n_art"], zscore=True)
        rows.append({"reading": reading_name(r), "network": net, "b across raw": f.params["across"], "b across z": fz.params["across"],
                     "t across": f.tvalues["across"], "p across": f.pvalues["across"], "b within z": fz.params[wv], "t within": f.tvalues[wv], "p within": f.pvalues[wv]})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("paper: ToM across b = -.4, t = -5.5, within t = 1.5 (p = .14); Pain across b = -.51, t = -7.4, within t = 1.9 (p = .06)")

**Result: not reproduced.** With age and motion in the model, a stronger across-network anti-correlation predicts a more adult-like timecourse in both networks (ToM t = -4.52, p = 1.5e-05; Pain t = -3.55, p = .00055), with the paper's sign, but both t values are out of tolerance (-5.5 and -7.4; -4.37 to -4.85 and -3.55 to -4.10 under the readings). Within-ToM does not predict ToM maturity under the main reading (t = 1.53, p = .13), as in the paper; under O2b with O3b it does (t = 1.99, p = .049). Within-Pain does predict Pain maturity (t = 2.40, p = .018; p = .0054 to .018 under every reading), where the paper's effect was not significant (p = .06). The criterion requires the within-network effects to be not significant, so the claim is not reproduced.


In [ ]:
rows = []
for r in READINGS:
    c = IRC[r][0].loc[children.index].join(MAT)
    for net, wv in [("ToM", "wToM"), ("Pain", "wPain")]:
        f = sb.ols(c, wv, [f"mat_{net}", "Age", "n_art"]); fz = sb.ols(c, wv, [f"mat_{net}", "Age", "n_art"], zscore=True)
        rows.append({"reading": reading_name(r), "measure": wv, "b maturity z": fz.params[f"mat_{net}"], "t maturity": f.tvalues[f"mat_{net}"],
                     "p maturity": f.pvalues[f"mat_{net}"], "b age z": fz.params["Age"], "t age": f.tvalues["Age"], "p age": f.pvalues["Age"]})
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))
print("paper: maturity t = .48 (ToM), .61 (Pain), n.s.; age t = 2.5 (p = .01), 1.96 (p = .05)")

**Result: reproduced, resting on an assumption.** Under the main reading neither maturity effect is significant (within-ToM t = .83, p = .41; within-Pain t = 1.80, p = .075), while age is (t = 2.39, p = .019; t = 4.22, p = 4.9e-05). Under O2b with O3b the Pain maturity effect has p = .040; the main reading is adopted (see deviations).

In [ ]:
c = children.join(MAT)
rr, pv, df = sb.partial_spearman(c.mat_ToM, c[TOM], c.n_art)
print(f"partial Spearman(ToM maturity, ToM score | motion) = {rr:.3f}, p = {pv:.2g}, df = {df} (paper .54, p = 1.3e-10)")
f = sb.ols(c, "mat_ToM", ["Age", TOM, "n_art"]); fz = sb.ols(c, "mat_ToM", ["Age", TOM, "n_art"], zscore=True)
print(f"regression: ToM score b z = {fz.params[TOM]:.2f}, t = {f.tvalues[TOM]:.2f}, p = {f.pvalues[TOM]:.3f}; age b z = {fz.params['Age']:.2f}, t = {f.tvalues['Age']:.2f}, p = {f.pvalues['Age']:.2g}")
print("paper: ToM score b = .1, t = 1.2, p = .3; age b = .5, t = 4.2, p < .00005")

**Result: reproduced.** ToM maturity correlates with ToM score controlling for motion (partial rs = .498, p = 6.1e-09; paper .54, tolerance .054). With age in the regression the ToM score effect vanishes (t = .04, p = .97) while age remains (t = 4.46, p = 1.8e-05).

In [ ]:
c = young.loc[pf_idx].join(MAT).copy(); c["fail"] = (c.FB_Group == "fail").astype(float)
f = sb.ols(c, "mat_ToM", ["fail", "Age", "n_art"]); fz = sb.ols(c, "mat_ToM", ["fail", "Age", "n_art"], zscore=True)
print(c.groupby("FB_Group").mat_ToM.agg(["mean", "sem", "count"]).round(3).to_string())
print(f"FB group (fail = 1): b raw = {f.params['fail']:.3f}, b z = {fz.params['fail']:.2f}, t = {f.tvalues['fail']:.2f}, p = {f.pvalues['fail']:.2f} (paper b = -.12, t = -.35, p = .73)")

**Result: reproduced.** ToM maturity does not differ between passers and failers with age and motion in the model (t = 1.53, p = .13; failers .095, passers .168).

## Face regions

### face-rc and face-overlap

Bilateral fusiform ROIs (deposited rFFA, 1019 voxels, and lFFA, 531 voxels).

In [ ]:
r_all = group_r("FFA", children.index); r_3 = group_r("FFA", y3)
print(f"children (n = {len(children)}) against adults: r = {r_all[0]:.3f}, p = {r_all[1]:.1e} (paper .86, p < 1e-47)")
print(f"3yo (n = {len(y3)}) against adults: r = {r_3[0]:.3f}, p = {r_3[1]:.1e} (paper .72, p < 1e-26)")
rows = []
for span in ["movie", "all"]:
    for k, (alpha, fdr) in O5.items():
        tab, ev, rk = RC[("FFA", span, k)]
        e3 = sb.reverse_correlation(NZ[("FFA", span)].loc[y3].values, alpha, fdr)[1]
        rows.append({"O6": span, "O5": k, "adult events": len(ev), "adult s": 2 * sum(map(len, ev)), "adult lengths (TRs)": [len(e) for e in rk.trs],
                     "3yo events": len(e3), "3yo s": 2 * sum(map(len, e3)), "3yo TRs in adult events": len(tr_set(e3) & tr_set(ev)), "3yo TR count": len(tr_set(e3))})
face = pd.DataFrame(rows)
print(face.to_string())
print("paper: adults 7 events, 88 s, lengths 16, 4, 5, 7, 4, 4, 4 TRs; 3yo 4 events, 18 s, 8 of 9 TRs in adult events")
tab, ev, rk = RC[("FFA", BEST_SPAN, BEST_O5)]
rows = []
for name, src in {"O7a z": NZ[("FFA", BEST_SPAN)], "O7b raw": RAW["FFA"]}.items():
    for i, row in rk.iterrows():
        c = children.join(src[row.peak_tr - 1].rename("mag")).dropna(subset=["mag"])
        rr, pv, _ = sb.partial_spearman(c.mag, c.Age, c.n_art)
        rows.append({"reading": name, "event": f"F{i + 1:02d}", "peak TR": row.peak_tr, "n": len(c), "rs": rr, "p": pv,
                     "p < .0071": pv < .0071, f"p < .05/{len(rk)}": pv < .05 / len(rk)})
print(f"face-event magnitudes against age (events of the {BEST_SPAN}, {BEST_O5} reading):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("paper: |rs| < .24, ps > .01, none significant at .0071")

**Result: not reproduced.** The children's mean fusiform timecourse correlates with the adults' (r = .818, within 10% of .86, p = 2.8e-39), but the 3yo correlation is .589 against .72, outside tolerance and above the paper's bound (p = 3.8e-16 against p < 1e-26). Reverse correlation finds 10 adult face events (108 s) and 6 three-year-old events (32 s) under the rc-events reading (O5a, O6b), against 7 (88 s) and 4 (18 s); no reading gives 7 adult events. No face-event magnitude changes with age at alpha .0071 (smallest p = .010), as in the paper.

In [ ]:
rng_face = np.random.default_rng(88)
rows = []
for span, k in [(BEST_SPAN, BEST_O5)] + [("movie", k) for k in O5 if k != BEST_O5 or BEST_SPAN != "movie"]:
    ef = RC[("FFA", span, k)][1]
    for other in ["ToM", "Pain"]:
        eo = RC[(other, span, k)][1]
        nf, no = sum(map(len, ef)), sum(map(len, eo))
        obs = len(tr_set(ef) & tr_set(eo))
        perm = sb.overlap_permutation([len(e) for e in ef], [len(e) for e in eo], 158 - nf, 158 - no, 1000, rng_face)
        rows.append({"reading": f"{span}, {k}", "face vs": other, "observed overlap TRs": obs, "perms <= observed": int((perm <= obs).sum()),
                     "p (fewer than chance)": float((perm <= obs).mean()), "mean perm overlap": perm.mean()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("paper: face-ToM 4 TRs, p = .11; face-pain 15 TRs, p = .93")

**Result: reproduced.** The face events overlap the ToM events by 7 TRs and the pain events by 25 TRs (paper 4 and 15), and neither overlap is smaller than chance (p = .052 and .983 under the rc-events reading; .098 to .166 and .94 to .96 under the others).

## Blocked claims

- replication-sample: the independent sample of 20 adults (Fedorenko and colleagues) is not in
  the snapshot, and the paper names no archive for it.
- hand-coded-events: the "mental", "social" and "pain" coding of Jacoby et al. (2016) is not in
  the snapshot (its README lists only the reverse-correlation events), and the paper does not
  give the coded times. The Saxe lab's web page offers the experiment script (movloc.zip,
  version of 29 March 2019) that holds them, with no DOI or repository commit to pin it, so it
  cannot be used as a pinned input.


In [ ]:
# replication-sample: the dataset's README lists the events "that replicated across the two
# samples" (onset TR; duration in TRs). Which Supplementary Table 3 events each overlaps. This
# restates the authors' result; it is not the second sample's data.
import re
readme = sb.text("README")
blk = readme.split("TRs identified by RC analysis")[1].split("TIMING OF EVENTS")[0]
tom_txt, pain_txt = blk.split("Pain Event Onsets")
def parse(s):
    return [tuple(int(v) for v in line.split(";")) for line in s.splitlines() if re.fullmatch(r"\s*\d+;\s*\d+\s*", line)]
for net, part in [("ToM", tom_txt), ("Pain", pain_txt)]:
    listed = parse(part)
    tab = sb.SUPP_TABLE3[sb.SUPP_TABLE3.network == net]
    hit = {ev: [(o, d) for o, d in listed if set(range(o, o + d)) & set(trs)] for ev, trs in zip(tab.event, tab.trs)}
    print(f"{net}: {len(listed)} replicated events listed in the README:", listed)
    print("   table events overlapped:", [ev for ev, h in hit.items() if h], "; not overlapped:", [ev for ev, h in hit.items() if not h])


The README's replicated events overlap all seven ToM events and nine of the twelve pain events of
Supplementary Table 3, all but P08, P09 and P11, as the replication-sample claim states, and they
are consistent with the onset mapping (stimulus seconds / 2 + 8) used for the table. This restates
the authors' result and does not test it on the second sample's data, so the claim stays blocked.


## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| participants | reproduced |
| supp-table1-behavior-by-age | reproduced |
| beh-tom-age | reproduced |
| beh-tom-dccs | reproduced |
| beh-fb-groups | reproduced |
| beh-fb-tom | reproduced |
| beh-fb-age | reproduced |
| beh-fb-dccs | reproduced |
| motion-child-adult | reproduced |
| motion-not-age-tom | reproduced |
| motion-not-fb | reproduced |
| irc-adult | partially reproduced |
| irc-age-within | not reproduced |
| irc-age-across | not reproduced |
| irc-adult-vs-child | partially reproduced |
| irc-control-networks | partially reproduced |
| irc-williams | not reproduced |
| irc-3yo-distinct | not reproduced |
| irc-3yo-not-anticorrelated | partially reproduced |
| irc-3yo-homologous | not reproduced |
| irc-3yo-nonbilateral | partially reproduced |
| irc-tom-behavior | partially reproduced |
| irc-tom-behavior-age | not reproduced (assumed reading) |
| irc-fb-within | not reproduced |
| irc-fb-within-dccs | not reproduced |
| irc-fb-across | not reproduced (assumed reading) |
| rc-events | partially reproduced |
| rc-overlap | partially reproduced |
| rc-child-adult-timecourse | not reproduced |
| supp-table4-other-networks | not reproduced |
| rc-age-events | not reproduced |
| rc-t04-tom | not reproduced |
| rc-fb-no-difference | reproduced (assumed reading) |
| rc-3yo-events | partially reproduced (assumed reading) |
| maturity-across | not reproduced |
| maturity-within | reproduced (assumed reading) |
| maturity-tom-behavior | reproduced |
| maturity-fb | reproduced |
| face-rc | not reproduced |
| face-overlap | reproduced |
| replication-sample | blocked: the independent adult sample is not public |
| hand-coded-events | blocked: the hand coding of Jacoby et al. 2016 is in no pinned public source |

How the analysis differs from the paper's. The analysis starts from the paper's own
preprocessed images, artifact indicators, CompCor regressors and ROI masks, all deposited in the
snapshot, so the differences are in the steps after them: the ROI-mean shortcut (exact for the
linear steps the Methods describe), SPM's discrete cosine high-pass filter for "a cutoff of 100
s", an intercept in the nuisance regression, the ppcor forms of the partial correlations, and the
readings O1 to O9 of the claims file, each run. The paper's analysis code is not public.

What was found about the dataset. Every behavioural and demographic number of the paper is
recovered from participants.tsv: the counts exactly, the other values within tolerance (for example
mean ToM scores .517 and .610 against .54 and .63, DCCS p .048 against .049, Inc-Fail p .015
against .01, DCCS b .148 against .16), beh-fb-tom under reading (a) only, and the artifact counts match the Methods to the
decimal. The deposited ROI masks have the sizes of Supplementary Table 2 and are in MNI mm, centered on the stated coordinates, and the artifact
indicators line up with spikes in the image data (median deviation 6.45 at the flagged
timepoints against at most 2.6 one or two TRs away). With these data and the pipeline of the
Methods, the adult inter-region correlations are higher within networks and less negative across
networks than the paper reports (.604, .398, -.051 against .48, .35, -.17); in the 3-year-olds
within-ToM is higher (.326 against .21) but within-Pain is lower (.179 against .23, and
non-bilateral Pain .146 against .17), and the
event-level tests of age and ToM score (rc-age-events, rc-t04-tom) and the false-belief group
effect on within-ToM correlation do not come out as in the paper. The voxel-size column
(named "Scanlog: Voxel slize") counts 2 seven-year-olds and 21 8-12-year-olds at 3.13 mm with
no slice gap, where the Methods say 3 and 20, and the no-gap level is stored as an empty value
where participants.json defines "None". The face-ROI coverage count differs from Supplementary
Fig. 4 (19 participants with any face-ROI voxel without data, against 17).